In [ ]:
!pip -q install transformers accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
set_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
# ---------- Chatbot 1: DialoGPT (conversational GPT-2) ----------
tok = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium").to(device).eval()

class DialoBot:
    def __init__(self):
        self.history = None

    def reply(self, text):
        new_ids = tok.encode(text + tok.eos_token, return_tensors="pt").to(device)
        inp = new_ids if self.history is None else torch.cat([self.history, new_ids], dim=-1)
        with torch.no_grad():
            self.history = model.generate(inp, attention_mask=torch.ones_like(inp), max_length=1000,
                                          pad_token_id=tok.eos_token_id, do_sample=True,
                                          top_k=50, top_p=0.9, temperature=0.7)
        return tok.decode(self.history[0, inp.shape[-1]:], skip_special_tokens=True)

bot = DialoBot()
for msg in ["Hello, how are you?", "What do you like to do in your free time?",
            "I am a student and I love programming.", "Can you give me a tip for exams?"]:
    print("You:", msg)
    print("Bot:", bot.reply(msg), "\n")

In [ ]:
# ---------- Interactive chat (type 'quit' to stop) ----------
bot = DialoBot()
while True:
    user = input("You: ")
    if user.strip().lower() in ["quit", "exit", "bye"]:
        print("Bot: Goodbye!")
        break
    print("Bot:", bot.reply(user))

In [ ]:
# ---------- Chatbot 2: instruction-tuned small LLM (ChatGPT-style) ----------
name = "Qwen/Qwen2.5-0.5B-Instruct"
tok2 = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype="auto").to(device).eval()

messages = [{"role": "system", "content": "You are a helpful assistant for college students. Keep answers short."}]

def chat(user_text, max_new_tokens=120):
    messages.append({"role": "user", "content": user_text})
    prompt = tok2.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tok2(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9)
    answer = tok2.decode(out[0, enc["input_ids"].shape[-1]:], skip_special_tokens=True)
    messages.append({"role": "assistant", "content": answer})
    return answer

for q in ["What is a transformer in NLP?", "Explain it in one simple example.", "Suggest 3 NLP project ideas."]:
    print("You:", q)
    print("Bot:", chat(q), "\n")

In [ ]:
# ---------- Interactive chat with the instruction model ----------
while True:
    user = input("You: ")
    if user.strip().lower() in ["quit", "exit", "bye"]:
        print("Bot: Goodbye!")
        break
    print("Bot:", chat(user))